# ROGII - Wellbore Geology Prediction

Kaggle-only EDA notebook for the ROGII code competition. It assumes the competition dataset is attached at `/kaggle/input/rogii-wellbore-geology-prediction`.

The notebook starts with exploratory analysis, then ends with a simple valid carry-forward baseline that writes `/kaggle/working/submission.csv`.

In [ ]:
from pathlib import Path
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

warnings.filterwarnings('ignore')
pd.set_option('display.max_columns', 120)
pd.set_option('display.max_rows', 80)
sns.set_theme(style='whitegrid', context='notebook', palette='viridis')
VIRIDIS = sns.color_palette('viridis', as_cmap=True)
VIRIDIS_COLORS = sns.color_palette('viridis', 8)

DATA_ROOT = Path('/kaggle/input/rogii-wellbore-geology-prediction')
WORK_DIR = Path('/kaggle/working')
SUBMISSION_PATH = WORK_DIR / 'submission.csv'

print('DATA_ROOT:', DATA_ROOT)
print('Exists:', DATA_ROOT.exists())

## 1. Discover Files

The competition data is organized by well. Each well has a horizontal-well CSV and a typewell CSV; `sample_submission.csv` tells us which horizontal-well row indices need predictions.

In [ ]:
def find_files(root: Path, pattern: str):
    return sorted(root.rglob(pattern)) if root.exists() else []


def well_name_from_horizontal_path(path: Path) -> str:
    return path.name.split('__horizontal_well.csv')[0]


def well_name_from_typewell_path(path: Path) -> str:
    return path.name.split('__typewell.csv')[0]


def parse_submission_id(value):
    well, row = str(value).rsplit('_', 1)
    return well, int(row)


def get_column(df: pd.DataFrame, name: str):
    lookup = {col.lower(): col for col in df.columns}
    return lookup.get(name.lower())


train_dir = DATA_ROOT / 'train'
test_dir = DATA_ROOT / 'test'
sample_path = DATA_ROOT / 'sample_submission.csv'

train_horizontal_files = find_files(train_dir, '*__horizontal_well.csv')
train_typewell_files = find_files(train_dir, '*__typewell.csv')
test_horizontal_files = find_files(test_dir, '*__horizontal_well.csv')
test_typewell_files = find_files(test_dir, '*__typewell.csv')
png_files = find_files(DATA_ROOT, '*.png')

inventory = pd.DataFrame([
    {'split': 'train', 'file_type': 'horizontal_well', 'count': len(train_horizontal_files)},
    {'split': 'train', 'file_type': 'typewell', 'count': len(train_typewell_files)},
    {'split': 'test', 'file_type': 'horizontal_well', 'count': len(test_horizontal_files)},
    {'split': 'test', 'file_type': 'typewell', 'count': len(test_typewell_files)},
    {'split': 'all', 'file_type': 'png', 'count': len(png_files)},
    {'split': 'all', 'file_type': 'sample_submission', 'count': int(sample_path.exists())},
])
display(inventory)
print('First train wells:', [well_name_from_horizontal_path(p) for p in train_horizontal_files[:5]])
print('First test wells:', [well_name_from_horizontal_path(p) for p in test_horizontal_files[:5]])

In [ ]:
sample_submission = pd.read_csv(sample_path)
id_col = sample_submission.columns[0]
target_col = 'tvt' if 'tvt' in sample_submission.columns else sample_submission.columns[-1]

submission_index = sample_submission[id_col].map(parse_submission_id)
sample_submission['well'] = [x[0] for x in submission_index]
sample_submission['row_idx'] = [x[1] for x in submission_index]

display(sample_submission.head())
print('sample_submission shape:', sample_submission.shape)
display(sample_submission.groupby('well')['row_idx'].agg(['min', 'max', 'count']).head(10))

## 2. Load Lightweight Metadata

The dataset is large enough that it is nicer to start with per-file summaries, then load detailed rows only for selected wells.

In [ ]:
def summarize_horizontal_file(path: Path, split: str):
    well = well_name_from_horizontal_path(path)
    df = pd.read_csv(path)
    tvt_input_col = get_column(df, 'TVT_input')
    tvt_col = get_column(df, 'TVT')
    md_col = get_column(df, 'MD')
    gr_col = get_column(df, 'GR')

    row = {
        'split': split,
        'well': well,
        'rows': len(df),
        'n_columns': df.shape[1],
        'columns': tuple(df.columns),
        'has_tvt': tvt_col is not None,
        'has_tvt_input': tvt_input_col is not None,
        'md_min': pd.to_numeric(df[md_col], errors='coerce').min() if md_col else np.nan,
        'md_max': pd.to_numeric(df[md_col], errors='coerce').max() if md_col else np.nan,
        'gr_mean': pd.to_numeric(df[gr_col], errors='coerce').mean() if gr_col else np.nan,
        'gr_std': pd.to_numeric(df[gr_col], errors='coerce').std() if gr_col else np.nan,
    }

    if tvt_input_col:
        tvt_input = pd.to_numeric(df[tvt_input_col], errors='coerce')
        hidden_mask = tvt_input.isna()
        row['tvt_input_known'] = int(tvt_input.notna().sum())
        row['tvt_input_missing'] = int(hidden_mask.sum())
        row['tvt_input_missing_frac'] = float(hidden_mask.mean())
        row['first_missing_row'] = int(np.argmax(hidden_mask.to_numpy())) if hidden_mask.any() else np.nan
        row['last_known_tvt_input'] = float(tvt_input.ffill().iloc[-1]) if tvt_input.notna().any() else np.nan
    else:
        row['tvt_input_known'] = 0
        row['tvt_input_missing'] = np.nan
        row['tvt_input_missing_frac'] = np.nan
        row['first_missing_row'] = np.nan
        row['last_known_tvt_input'] = np.nan

    if tvt_col:
        tvt = pd.to_numeric(df[tvt_col], errors='coerce')
        row['tvt_min'] = tvt.min()
        row['tvt_max'] = tvt.max()
        row['tvt_range'] = tvt.max() - tvt.min()
    else:
        row['tvt_min'] = np.nan
        row['tvt_max'] = np.nan
        row['tvt_range'] = np.nan

    return row


def summarize_typewell_file(path: Path, split: str):
    well = well_name_from_typewell_path(path)
    df = pd.read_csv(path)
    tvt_col = get_column(df, 'TVT')
    gr_col = get_column(df, 'GR')
    geology_col = get_column(df, 'Geology')
    return {
        'split': split,
        'well': well,
        'rows': len(df),
        'n_columns': df.shape[1],
        'columns': tuple(df.columns),
        'tvt_min': pd.to_numeric(df[tvt_col], errors='coerce').min() if tvt_col else np.nan,
        'tvt_max': pd.to_numeric(df[tvt_col], errors='coerce').max() if tvt_col else np.nan,
        'gr_mean': pd.to_numeric(df[gr_col], errors='coerce').mean() if gr_col else np.nan,
        'gr_std': pd.to_numeric(df[gr_col], errors='coerce').std() if gr_col else np.nan,
        'n_geology_labels': df[geology_col].nunique() if geology_col else np.nan,
    }


horizontal_meta = pd.DataFrame(
    [summarize_horizontal_file(p, 'train') for p in train_horizontal_files]
    + [summarize_horizontal_file(p, 'test') for p in test_horizontal_files]
)
typewell_meta = pd.DataFrame(
    [summarize_typewell_file(p, 'train') for p in train_typewell_files]
    + [summarize_typewell_file(p, 'test') for p in test_typewell_files]
)

display(horizontal_meta.head())
display(typewell_meta.head())

## 3. Schema And Missingness

Train horizontal files include target `TVT`; test horizontal files expose `TVT_input`, which is blank in the evaluation interval.

In [ ]:
def column_presence(files, split, well_parser):
    rows = []
    for path in files:
        well = well_parser(path)
        df = pd.read_csv(path, nrows=5)
        for col in df.columns:
            rows.append({'split': split, 'well': well, 'column': col})
    return pd.DataFrame(rows)


horizontal_columns = pd.concat([
    column_presence(train_horizontal_files, 'train', well_name_from_horizontal_path),
    column_presence(test_horizontal_files, 'test', well_name_from_horizontal_path),
], ignore_index=True)

column_summary = (
    horizontal_columns.groupby(['split', 'column'])['well']
    .nunique()
    .reset_index(name='well_count')
    .sort_values(['split', 'well_count', 'column'], ascending=[True, False, True])
)
display(column_summary)

if not horizontal_meta.empty:
    display(horizontal_meta.groupby('split')[['rows', 'tvt_input_missing_frac', 'tvt_range', 'gr_mean', 'gr_std']].describe().T)

In [ ]:
if not horizontal_meta.empty:
    fig, axes = plt.subplots(1, 3, figsize=(18, 4))
    sns.histplot(data=horizontal_meta, x='rows', hue='split', bins=30, ax=axes[0], element='step', palette='viridis')
    axes[0].set_title('Rows per horizontal well')
    sns.histplot(data=horizontal_meta, x='tvt_input_missing_frac', hue='split', bins=30, ax=axes[1], element='step', palette='viridis')
    axes[1].set_title('Hidden TVT_input fraction')
    sns.scatterplot(data=horizontal_meta, x='rows', y='tvt_input_missing', hue='split', ax=axes[2], palette='viridis')
    axes[2].set_title('Hidden rows by well length')
    plt.tight_layout()
    plt.show()

if not typewell_meta.empty:
    fig, axes = plt.subplots(1, 2, figsize=(14, 4))
    sns.histplot(data=typewell_meta, x='rows', hue='split', bins=30, ax=axes[0], element='step', palette='viridis')
    axes[0].set_title('Rows per typewell')
    sns.histplot(data=typewell_meta, x='n_geology_labels', hue='split', bins=20, ax=axes[1], element='step', palette='viridis')
    axes[1].set_title('Geology labels per typewell')
    plt.tight_layout()
    plt.show()

## 4. Inspect Sample Wells

These plots show the horizontal well GR curve, the known/hidden `TVT_input` track, and the matching typewell GR-vs-TVT reference.

In [ ]:
horizontal_lookup = {
    **{well_name_from_horizontal_path(p): p for p in train_horizontal_files},
    **{well_name_from_horizontal_path(p): p for p in test_horizontal_files},
}
typewell_lookup = {
    **{well_name_from_typewell_path(p): p for p in train_typewell_files},
    **{well_name_from_typewell_path(p): p for p in test_typewell_files},
}


def load_well(well):
    horizontal = pd.read_csv(horizontal_lookup[well])
    typewell = pd.read_csv(typewell_lookup[well]) if well in typewell_lookup else None
    return horizontal, typewell


def plot_well(well):
    horizontal, typewell = load_well(well)
    md_col = get_column(horizontal, 'MD')
    gr_col = get_column(horizontal, 'GR')
    tvt_input_col = get_column(horizontal, 'TVT_input')
    tvt_col = get_column(horizontal, 'TVT')

    x = pd.to_numeric(horizontal[md_col], errors='coerce') if md_col else horizontal.index
    fig, axes = plt.subplots(1, 3, figsize=(19, 4))

    if gr_col:
        axes[0].plot(x, pd.to_numeric(horizontal[gr_col], errors='coerce'), lw=1, color=VIRIDIS_COLORS[2])
    axes[0].set_title(f'{well}: horizontal GR')
    axes[0].set_xlabel('MD' if md_col else 'row')
    axes[0].set_ylabel('GR')

    if tvt_input_col:
        tvt_input = pd.to_numeric(horizontal[tvt_input_col], errors='coerce')
        axes[1].plot(x, tvt_input, lw=1, label='TVT_input', color=VIRIDIS_COLORS[4])
        if tvt_input.isna().any():
            hidden_start = int(np.argmax(tvt_input.isna().to_numpy()))
            axes[1].axvline(x.iloc[hidden_start] if hasattr(x, 'iloc') else hidden_start, color=VIRIDIS_COLORS[7], ls='--', lw=1, label='first hidden row')
    if tvt_col:
        axes[1].plot(x, pd.to_numeric(horizontal[tvt_col], errors='coerce'), lw=1, alpha=0.65, label='TVT', color=VIRIDIS_COLORS[1])
    axes[1].invert_yaxis()
    axes[1].set_title('Horizontal TVT track')
    axes[1].set_xlabel('MD' if md_col else 'row')
    axes[1].legend(loc='best')

    if typewell is not None:
        tw_tvt_col = get_column(typewell, 'TVT')
        tw_gr_col = get_column(typewell, 'GR')
        geology_col = get_column(typewell, 'Geology')
        if tw_tvt_col and tw_gr_col:
            axes[2].plot(pd.to_numeric(typewell[tw_gr_col], errors='coerce'), pd.to_numeric(typewell[tw_tvt_col], errors='coerce'), lw=1, color=VIRIDIS_COLORS[3])
            axes[2].invert_yaxis()
        if geology_col:
            label_counts = typewell[geology_col].value_counts().head(8)
            axes[2].text(0.98, 0.02, '\n'.join([f'{k}: {v}' for k, v in label_counts.items()]), transform=axes[2].transAxes, ha='right', va='bottom', fontsize=8)
    axes[2].set_title('Typewell GR vs TVT')
    axes[2].set_xlabel('GR')
    axes[2].set_ylabel('TVT')

    plt.tight_layout()
    plt.show()


sample_wells = list(sample_submission['well'].drop_duplicates().head(3))
if len(sample_wells) < 3:
    sample_wells += [w for w in horizontal_lookup if w not in sample_wells][:3 - len(sample_wells)]

for well in sample_wells:
    if well in horizontal_lookup:
        plot_well(well)

## 5. Target And Feature Relationships In Train

This loads a manageable sample of training rows for relationships between `TVT`, `TVT_input`, `MD`, `Z`, and `GR`.

In [ ]:
def load_horizontal_sample(files, max_wells=60, rows_per_well=1500, split='train'):
    frames = []
    for path in files[:max_wells]:
        well = well_name_from_horizontal_path(path)
        df = pd.read_csv(path)
        if len(df) > rows_per_well:
            df = df.sample(rows_per_well, random_state=42).sort_index()
        df = df.copy()
        df['well'] = well
        df['split'] = split
        frames.append(df)
    return pd.concat(frames, ignore_index=True) if frames else pd.DataFrame()


train_sample = load_horizontal_sample(train_horizontal_files)
print('train sample shape:', train_sample.shape)
display(train_sample.head())

numeric_cols = [c for c in ['MD', 'X', 'Y', 'Z', 'GR', 'TVT', 'TVT_input', 'ANCC', 'ASTNU', 'ASTNL', 'EGFDU', 'EGFDL', 'BUDA'] if c in train_sample.columns]
if numeric_cols:
    display(train_sample[numeric_cols].describe().T)
    corr = train_sample[numeric_cols].apply(pd.to_numeric, errors='coerce').corr()
    plt.figure(figsize=(10, 8))
    sns.heatmap(corr, cmap='viridis', annot=False, square=True)
    plt.title('Train numeric correlation sample')
    plt.tight_layout()
    plt.show()

In [ ]:
if not train_sample.empty and 'TVT' in train_sample.columns:
    fig, axes = plt.subplots(1, 3, figsize=(18, 4))
    sns.histplot(pd.to_numeric(train_sample['TVT'], errors='coerce'), bins=60, ax=axes[0], color=VIRIDIS_COLORS[4])
    axes[0].set_title('Train TVT distribution')
    if 'GR' in train_sample.columns:
        sns.scatterplot(data=train_sample.sample(min(len(train_sample), 20000), random_state=42), x='GR', y='TVT', hue='well', legend=False, s=8, alpha=0.35, ax=axes[1], palette='viridis')
        axes[1].invert_yaxis()
        axes[1].set_title('GR vs TVT sample')
    if 'MD' in train_sample.columns:
        sns.scatterplot(data=train_sample.sample(min(len(train_sample), 20000), random_state=42), x='MD', y='TVT', hue='well', legend=False, s=8, alpha=0.35, ax=axes[2], palette='viridis')
        axes[2].invert_yaxis()
        axes[2].set_title('MD vs TVT sample')
    plt.tight_layout()
    plt.show()

## 6. EDA Notes To Revisit

- `TVT_input` defines the known history and the hidden future interval for each test well.
- Typewell `GR` vs `TVT` is the vertical reference signature; the horizontal `GR` curve is the signal to align against it.
- Train-only geology-top columns should not be used directly as inference features unless you build a separate auxiliary model that is also available at test time.
- A smooth carry-forward baseline is valid, but it will struggle where faults or abrupt stratigraphic shifts change TVT quickly.

## 7. Simple Baseline Submission

This baseline fills the hidden part of `TVT_input` with the last known TVT value for the well, then maps those row-level predictions to `sample_submission.csv`.

In [ ]:
def carry_forward_tvt(horizontal: pd.DataFrame) -> pd.Series:
    tvt_input_col = get_column(horizontal, 'TVT_input')
    tvt_col = get_column(horizontal, 'TVT')

    if tvt_input_col is not None:
        tvt = pd.to_numeric(horizontal[tvt_input_col], errors='coerce')
    elif tvt_col is not None:
        tvt = pd.to_numeric(horizontal[tvt_col], errors='coerce')
    else:
        tvt = pd.Series(np.nan, index=horizontal.index, dtype='float64')

    filled = tvt.ffill().bfill()
    if filled.isna().all():
        filled = pd.Series(0.0, index=horizontal.index)
    return filled.astype('float64')


def make_well_predictions(horizontal_files):
    predictions = {}
    for path in horizontal_files:
        well = well_name_from_horizontal_path(path)
        horizontal = pd.read_csv(path)
        predictions[well] = carry_forward_tvt(horizontal).reset_index(drop=True)
    return predictions


well_predictions = make_well_predictions(test_horizontal_files)
print('prediction wells:', len(well_predictions))

In [ ]:
def rmse(y_true, y_pred):
    y_true = np.asarray(y_true, dtype='float64')
    y_pred = np.asarray(y_pred, dtype='float64')
    mask = np.isfinite(y_true) & np.isfinite(y_pred)
    return float(np.sqrt(np.mean((y_true[mask] - y_pred[mask]) ** 2))) if mask.any() else np.nan


def carry_forward_validation(horizontal_files, tail_fraction=0.25, max_wells=40):
    scores = []
    for path in horizontal_files[:max_wells]:
        well = well_name_from_horizontal_path(path)
        df = pd.read_csv(path)
        tvt_col = get_column(df, 'TVT')
        tvt_input_col = get_column(df, 'TVT_input')
        if tvt_col is None:
            continue

        y = pd.to_numeric(df[tvt_col], errors='coerce')
        x = pd.to_numeric(df[tvt_input_col], errors='coerce') if tvt_input_col else y.copy()
        eval_start = int(len(df) * (1 - tail_fraction))
        x.iloc[eval_start:] = np.nan

        tmp = df.copy()
        tmp['TVT_input'] = x
        pred = carry_forward_tvt(tmp)
        scores.append({'well': well, 'rows': len(df), 'rmse': rmse(y.iloc[eval_start:], pred.iloc[eval_start:])})

    return pd.DataFrame(scores)


if train_horizontal_files:
    val_scores = carry_forward_validation(train_horizontal_files)
    display(val_scores.head())
    print('Mean well RMSE:', val_scores['rmse'].mean())
else:
    print('No train files found in this environment.')

In [ ]:
submission = sample_submission[[id_col]].copy()
predicted_tvt = []
missing_wells = set()

global_fallback = 0.0
all_known_values = [series.dropna().to_numpy() for series in well_predictions.values()]
if all_known_values:
    non_empty_values = [values for values in all_known_values if len(values)]
    if non_empty_values:
        global_fallback = float(np.nanmedian(np.concatenate(non_empty_values)))

for sub_id in sample_submission[id_col]:
    well, row_idx = parse_submission_id(sub_id)
    pred_series = well_predictions.get(well)
    if pred_series is None or len(pred_series) == 0:
        missing_wells.add(well)
        predicted_tvt.append(global_fallback)
    elif 0 <= row_idx < len(pred_series):
        predicted_tvt.append(float(pred_series.iloc[row_idx]))
    else:
        predicted_tvt.append(float(pred_series.iloc[-1]))

submission[target_col] = predicted_tvt
submission.to_csv(SUBMISSION_PATH, index=False)

print('Wrote:', SUBMISSION_PATH)
print('Rows:', len(submission))
print('Missing wells:', sorted(missing_wells)[:10], 'count=', len(missing_wells))
display(submission.head())
display(submission[target_col].describe())